# Building a crossword grid

One puzzle, from a list of words to a finished grid, stopping at each stage to
look at what the builder has in its hands.

The example is a British cryptic: a 15x15 blocked grid in which about half the
letters of every entry are unchecked, built around a theme. American and barred puzzles come out of the
same machinery with a different rule set and a different library.

1. Read a dictionary and turn it into something a search can use
2. Take a grid pattern from a library of published ones
3. Work out which patterns could hold the theme at all
4. Seat the theme words
5. Fill everything else
6. Read the result back as a puzzle

Run the cells in order.

## Setup

The notebook lives in `notebooks/`, so first point Python at the repository root.

In [1]:
import os
import sys
from collections import Counter

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

from crossword.render import show, cells_of, clue_lists

print("working from", os.getcwd())

working from /Users/angus/Documents/Crosswords/Crossword Builder


## 1. The words

`load` reads a word list and normalises it. Each entry keeps two forms.
`text` is what goes in the grid: lowercase, no punctuation or spaces. `surface`
is one original spelling, kept for display. The grid holds `twelfthnight`; the
enumeration a solver reads is (7,5), and that can only be recovered from the
surface form.

`proper` marks entries that were always capitalised. `phrase` marks any entry
whose text differs from its surface, which catches spaces, hyphens and
apostrophes together.

In [2]:
from crossword.words import load

entries = load("crossword/UKACD.txt")
print(f"{len(entries):,} entries\n")

for entry in entries[:3] + [e for e in entries if e.phrase][:2]:
    print(f"  text={entry.text!r:16} surface={entry.surface!r:16} "
          f"proper={entry.proper} phrase={entry.phrase}")

221,835 entries

  text='aardvark'       surface='aardvark'       proper=False phrase=False
  text='aardvarks'      surface='aardvarks'      proper=False phrase=False
  text='aardwolf'       surface='aardwolf'       proper=False phrase=False
  text='abadegg'        surface='a bad egg'      proper=False phrase=True
  text='abadhat'        surface='a bad hat'      proper=False phrase=True


## 2. The index: matching patterns with integers

The filler asks one question, millions of times: *given a slot with some
letters already known, which words still fit?*

Answering it by scanning the dictionary and comparing characters is far too
slow. Instead the index precomputes the answer as bit arithmetic.

**The idea.** Number the five-letter words 0, 1, 2, ... Then a *set* of words is
a single integer, where bit *i* is 1 if word *i* is in the set. For every
position and every letter, the index stores one such integer: the set of words
having that letter in that position.

    position 0, letter S  ->  ...0001000100101  (every five-letter word starting S)
    position 2, letter E  ->  ...0100010001001  (every one with E in the middle)

**Matching.** A pattern is the intersection of one set per known letter, and
intersection of bitsets is bitwise AND:

    S _ E _ L   ->   pos0_S  &  pos2_E  &  pos4_L

One AND per known letter, whatever the size of the dictionary, and the result
is the exact set of candidates. Counting them is `bit_count`, a single
operation. Python's integers are arbitrary-precision, so a 10,162-word set is
just a 10,162-bit number and the arithmetic is done in C.

**Why one universe per length.** Word ids are assigned within a length, so bit
17 means a different word in the five-letter index than in the six-letter one.
That costs nothing, because a slot of five cells can only ever be filled by a
five-letter word: the two universes never need to be compared.

In [3]:
from crossword.index import Index

index = Index(entries)
bucket = index.lengths[5]
print(f"{len(bucket.words):,} five-letter words\n")

mask = bucket.match("s.e.l")
print(f"S_E_L  ->  {bucket.count('s.e.l')} candidates")
print("       ", ", ".join(bucket.iterate(mask)))

# The mask really is one integer, one bit per word.
print(f"\nthe mask is an int of {mask.bit_length()} bits, "
      f"{bin(mask).count('1')} of them set")

10,162 five-letter words

S_E_L  ->  16 candidates
        sheal, sheel, shell, sheol, skell, smell, snell, speal, speel, spell, steal, steel, steil, stell, sweal, swell

the mask is an int of 8556 bits, 16 of them set


### What that buys

Compare the bitset against the obvious approach of testing every word.

In [4]:
import re
import time

pattern = "s.e.l"
words = bucket.words

began = time.time()
for _ in range(200):
    naive = [w for w in words if re.match("^" + pattern + "$", w)]
scan = (time.time() - began) / 200

began = time.time()
for _ in range(200):
    fast = list(bucket.iterate(bucket.match(pattern)))
bits = (time.time() - began) / 200

print(f"scanning 10,162 words: {scan * 1e6:8.0f} us")
print(f"bitset intersection:   {bits * 1e6:8.0f} us")
print(f"same answer: {sorted(naive) == sorted(fast)}")
print(f"speedup: {scan / bits:.0f}x")

scanning 10,162 words:     3755 us
bitset intersection:         11 us
same answer: True
speedup: 347x


That gap is the whole reason a fill completes in seconds. The search visits
tens of thousands of nodes and computes a candidate set for every empty slot at
each one, so this operation runs millions of times in a single puzzle.

## 3. The theme

The point of the project: fit as many words from a chosen list into one grid as
possible.

In [5]:
theme = ["kestrel", "redwing", "wheatear", "fieldfare",
         "goldcrest", "nuthatch", "siskin", "dunnock"]

for length, count in sorted(Counter(len(w) for w in theme).items()):
    print(f"  {count} word(s) of {length} letters")

  1 word(s) of 6 letters
  3 word(s) of 7 letters
  2 word(s) of 8 letters
  2 word(s) of 9 letters


## 4. The grid library

Grids are not invented here. They are taken from 120 patterns extracted from
8,348 published Guardian cryptics, geometry only.

The reason is empirical. Randomly generated patterns that satisfy every rule in
this project fill very badly; published ones fill immediately. Setters know
things about where blocks can go that none of the rules capture, and borrowing
their grids inherits all of it without having to state any of it.

In the drawing below, **black squares are blocks** and the small figures are
clue numbers.

In [6]:
from crossword import library

patterns = library.load()
first = patterns[0]
print(f"{len(patterns)} patterns; this one was used by "
      f"{first.uses} published puzzles")
show(first.grid())

120 patterns; this one was used by 458 published puzzles


### What the builder sees

It does not reason about squares. It reasons about *slots*: maximal runs of
white cells three or more long, which are the entries. A cell where an across
slot crosses a down slot is **checked**, meaning the solver gets two chances at
it; a white cell in only one slot is **unchecked**.

British grids check about half the letters of each entry, and in this lattice
checked and unchecked alternate strictly. Across the grid as a whole the
unchecked cells outnumber the checked ones, because the rows between the long
entries are unchecked all the way along.

In [7]:
grid = first.grid()
slots = grid.slots()
checked = grid.checked_cells()

blocks = len(grid.blocks)
white = grid.size ** 2 - blocks
print(f"{grid.size}x{grid.size} = {grid.size ** 2} cells")
print(f"  {blocks:3d} blocks")
print(f"  {len(checked):3d} white and checked   (in an across entry and a down entry)")
print(f"  {white - len(checked):3d} white and unchecked (in one direction only)")

print(f"\n{len(slots)} entries. How many of each length:")
for length, count in sorted(Counter(s.length for s in slots).items()):
    print(f"  {length:2d} letters: {count}")

print("\nC = checked, U = unchecked, # = block")
print(grid.annotate(gap=" "))

15x15 = 225 cells
   65 blocks
   58 white and checked   (in an across entry and a down entry)
  102 white and unchecked (in one direction only)

30 entries. How many of each length:
   4 letters: 4
   5 letters: 4
   6 letters: 4
   7 letters: 4
   8 letters: 4
   9 letters: 4
  10 letters: 4
  11 letters: 2

C = checked, U = unchecked, # = block
C U C U C U C # C U C U C U C
U # U # U # U # U # U # U # U
C U C U C # C U C U C U C U C
U # U # U # U # U # U # U # U
C U C U C U C U C U # U C U C
U # U # U # # # U # U # U # U
# # # # C U C U C U C U C U C
U # U # U # U # U # U # U # U
C U C U C U C U C U C # # # #
U # U # U # U # # # U # U # U
C U C U # U C U C U C U C U C
U # U # U # U # U # U # U # U
C U C U C U C U C # C U C U C
U # U # U # U # U # U # U # U
C U C U C U C # C U C U C U C


### The rules

Six predicates decide whether a pattern is one a setter would print. Each was
derived by measuring published puzzles rather than by reasoning about what
seems fair.

Take rule 4, the checked fraction. It requires at least half the letters of
every entry to be checked, rounding *down*: a nine-letter entry needs four, not
five. That rounding is not a detail. The commonest nine-letter pattern in
British cryptics is UCUCUCUCU, which has exactly four, and a rule demanding
five rejects a third of all published puzzles.

Below, the same predicate is run against a real pattern, and then against one
broken by adding a single block without its rotational partner.

In [8]:
from crossword.rules import validate, RuleSet, check_checked_fraction

print("published pattern:", validate(grid) or "clean")

# What rule 4 asks of each entry length present here.
import math
for length in sorted({s.length for s in slots}):
    need = math.floor(length * RuleSet().min_checked_fraction)
    print(f"  a {length:2d}-letter entry needs {need} checked letters")

broken = first.grid()
broken.blocks.add((0, 4))
broken._stamp += 1
broken._derived.clear()
print("\none block added without its partner:")
for problem in validate(broken)[:4]:
    print("  ", problem.rule, "-", problem.detail)

published pattern: clean
  a  4-letter entry needs 2 checked letters
  a  5-letter entry needs 2 checked letters
  a  6-letter entry needs 3 checked letters
  a  7-letter entry needs 3 checked letters
  a  8-letter entry needs 4 checked letters
  a  9-letter entry needs 4 checked letters
  a 10-letter entry needs 5 checked letters
  a 11-letter entry needs 5 checked letters

one block added without its partner:
   run_length - across at (0,5) length 2 is shorter than 3
   isolated_cell - cell (0, 5) is in no entry
   consecutive_unchecked - down at (0,6) length 5 has 2 unchecked cells in a row
   symmetry - block (0, 4) has no partner at (14, 10)


## 5. Which grids could hold the theme?

Before any search runs there is a free upper bound. A grid offering four
seven-letter entries cannot hold five seven-letter theme words, whatever the
search does. Matching the theme's lengths against each grid's supply gives a
*ceiling*, and patterns are tried in that order, best first.

In [9]:
from crossword import coverage

scored = sorted(((coverage.ceiling(p.profile(), theme), p) for p in patterns),
                key=lambda pair: -pair[0])
best = scored[0][0]
print(f"best ceiling in the library: {best} of {len(theme)} theme words")
print(f"patterns reaching it:        {sum(1 for c, _ in scored if c == best)}")

print("\ntheme needs      grid supplies")
supply = scored[0][1].profile()
for length, count in sorted(Counter(len(w) for w in theme).items()):
    print(f"  {count} x {length:2d}          {supply.get(length, 0)} entries")

best ceiling in the library: 8 of 8 theme words
patterns reaching it:        52

theme needs      grid supplies
  1 x  6          4 entries
  3 x  7          4 entries
  2 x  8          4 entries
  2 x  9          4 entries


## 6. Seating the theme, and filling the rest

Two searches, nested, and quite different in kind.

**Seating** is a branch-and-bound over which theme word goes in which slot.
Every placement can be undone, and a bound (words already seated, plus those
that could still be seated) cuts branches that cannot beat what has been found.

**Filling** is backtracking with forward checking. At every node it computes
the candidate set for each empty slot -- those bitset intersections again. That
single pass does two jobs: any empty set kills the node before a word is tried,
and the smallest set says which slot to expand next. Filling the most
constrained slot first is what keeps the tree narrow.

The fill search is the subject of [notebook 2](02-the-fill-search.ipynb).

In [10]:
import time

began = time.time()
result = coverage.best_over_library(
    patterns, index, theme, RuleSet(),
    time_limit=90, commonness=3.0, aim=0.85, seed=1,
)
print(f"{time.time() - began:.1f}s")
print(f"complete grid:      {result.ok}")
print(f"theme words seated: {result.n} of {len(theme)}  (ceiling {result.ceiling})")
print("seated:", ", ".join(result.placed))

0.1s
complete grid:      True
theme words seated: 8 of 8  (ceiling 8)
seated: fieldfare, goldcrest, nuthatch, wheatear, dunnock, kestrel, redwing, siskin


### The finished grid

The theme words are tinted.

In [11]:
finished = result.grid
marked = [cell for word in result.placed for cell in cells_of(finished, word)]
show(finished, highlight=marked)

## 7. When the library is not enough

That theme fitted a published grid outright, so the search stopped there. It
often does not, and then there is a second stage that the run above never
reached.

If no library pattern can hold the list, the builder breeds new ones: take the
best-fitting grids and flip a symmetric pair of blocks, keeping every mutant
that still passes the rules and still looks like a Guardian grid. That is a
hill-climb whose score is *how many theme words actually get seated*, not how
closely the grid's lengths match the list. Scoring on lengths was tried first
and made results worse: a grid can offer four seven-letter entries and still
have no way to fit four seven-letter words through each other.

It runs as a fallback and never as a merge. The library's own answer is
computed first and kept unless breeding beats it outright, so the two arms can
only win against each other, never lose.

In [12]:
from crossword import mutate

seed_pattern = scored[0][1]
options = list(mutate.neighbours(seed_pattern, RuleSet()))
print(f"{len(options)} legal grids are one symmetric block-pair flip away")

before = seed_pattern.grid()
# Prefer a flip that changes the entry count, which shows what a coarse move
# this is: removing one block can merge two entries into a much longer one.
flipped, after = next(
    ((cell, g) for cell, g in options if len(g.slots()) != len(before.slots())),
    options[0],
)
changed = before.blocks ^ after.blocks
print(f"flipping {flipped} moves {len(changed)} cells: {sorted(changed)}")
print(f"entries: {len(before.slots())} -> {len(after.slots())}")
print(f"longest entry: {max(s.length for s in before.slots())} -> "
      f"{max(s.length for s in after.slots())}")
show(after, highlight=changed)

29 legal grids are one symmetric block-pair flip away
flipping (0, 7) moves 2 cells: [(0, 7), (14, 7)]
entries: 30 -> 28
longest entry: 11 -> 15


`make_grid.py` runs this automatically whenever a word list is given. In the
notebook `best_over_library` was called directly, which is the library arm on
its own; `mutate.best_with_tailoring` is the pair of them.

## 8. Reading it back as a puzzle

A cell earns a number when an entry starts there in *either* direction, and a
cell that starts both shares one number between them. That sharing is why the
across and down lists have gaps rather than each running 1, 2, 3.

In [13]:
across, down = clue_lists(finished)

for name, items in (("ACROSS", across), ("DOWN", down)):
    print(name)
    for number, answer in items[:6]:
        print(f"  {number:>3}  {answer.upper()}")
    print("   ...")

ACROSS
    1  RUNITSELF
    6  KAGO
    8  NUTHATCH
    9  ENSILE
   10  ISOPOD
   11  INDIRECT
   ...
DOWN
    1  ROUES
    2  NOHOPER
    3  TOTED
    4  ECHOISM
    5  FIELDFARE
    6  KESTREL
   ...


### Out to a file

`to_ipuz` and `to_exolve` write the two formats setters use: ipuz for Exet and
most desktop software, Exolve for a self-contained web page. Enumerations come
from the surface forms, which is what the distinction at the top was for.

Barred grids take a third route. Neither format can place a bar, so
`export.write_html` draws a printable page instead.

In [14]:
from crossword import export

surfaces = {e.text: e.surface for e in entries}
document = export.to_ipuz(finished, title="Dawn Chorus", surfaces=surfaces)
print("dimensions:", document["dimensions"])
for clue in document["clues"]["Across"][:3]:
    print("  ", clue)

dimensions: {'width': 15, 'height': 15}
   {'number': '1', 'label': '1', 'clue': '(3,6)', 'answer': 'RUN ITSELF'}
   {'number': '6', 'label': '6', 'clue': '(4)', 'answer': 'KAGO'}
   {'number': '8', 'label': '8', 'clue': '(8)', 'answer': 'NUTHATCH'}


## Next

- **[the fill search](02-the-fill-search.ipynb)** in detail: forward checking,
  which entry to expand next, how word familiarity is scored so that a grid
  does not fill itself with ISLE and OVER, and what a failing search looks like
- **the other styles**: American and barred, and what had to change for each
- **ninas and pangrams**: constraints a setter adds on top

`BASELINE.md` records what each change to the builder was measured to be
worth.